### Procedure

1. `05.dataset` 디렉터리의 양성/음성 데이터셋을 불러온다.

2. 불러온 데이터셋을 스케일링 한다.

3. (train, val), test 데이터셋을 8:2 로 patient-wise 분리한다.

4. train 데이터셋은 sample-wise로 1:1 balancing 된다. 
  
   - 이 경우 moghadam의 것과 사실상 차별점이 없어지게 된다.
   - 환자를 저혈압 발생 환자와 아닌 환자로 구분할 필요가 있는가? 이 부분은 실험이 필요할 듯 하다. spass_paper에서는 말고.
     지금은 구분하지 말자. 저혈압 환자의 비저혈압 패턴과 비저혈압 환자의 비저혈압 패턴이 다소 차이가 있을 수 있으므로 모델이
     폭 넓게 학습할 가능성이 있기 때문이다.

5. InceptionTime, MLP, CNN, ResNet 모델로 학습시킨다.

In [2]:
import datetime
from pathlib import Path
import numpy as np
import numpy.typing as npt
from src.module.utils import parse_args, load_config, load_npy, save_npy
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import logging

from keras.models import Model
from keras.layers import Dense
from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from sktime.classification.deep_learning import InceptionTimeClassifier
from sklearn.metrics import roc_auc_score, classification_report

In [2]:
from dataclasses import dataclass


@dataclass
class Arguments:
    conf: str = "../../config/conf.yaml"
    strategy: str = "hypophetversion2"
    min = None
    max = None

In [3]:
def set_environment(args, config, model_name) -> dict:
    data_path = Path(config["data_path"]).expanduser()
    src_path = data_path / "05.dataset"
    pos_path = np.array(
        sorted(src_path.rglob(f"{args.strategy}_adj{config['adjacency']}_pos_dataset*"))
    )
    neg_path = np.array(
        sorted(src_path.rglob(f"{args.strategy}_adj{config['adjacency']}_neg_dataset*"))
    )

    save_dir = Path(config["model_path"]).expanduser()
    save_dt = datetime.datetime.now().strftime("%y%m%d_%H%M%S")
    save_path = save_dir / (save_dt + f"_{args.strategy}") / f"{model_name}.model.keras"
    env = {
        "src_path": src_path,
        "pos_path": pos_path,
        "neg_path": neg_path,
        "save_path": save_path,
    }
    return env


def scale_input(input) -> npt.NDArray:
    return input.astype(np.float32) / 200


def split_array(arr, ratio, seed) -> tuple:
    train, test = train_test_split(arr, test_size=ratio, random_state=seed)
    train, val = train_test_split(train, test_size=ratio, random_state=seed)
    logging.info(
        f"total(train/val/test): {arr.size}({train.size}/{val.size}/{test.size}) cases"
    )
    return train, val, test


def extract_dataset(x, y, idx):
    extract_x = np.expand_dims(x[idx], 2)
    extract_y = y[idx]
    print(extract_y.shape)
    logging.info(
        f"X: {extract_x.shape} / y(pos/neg): {extract_y.size}({sum(extract_y==1)[0]}/{sum(extract_y==0)[0]})"
    )
    return extract_x, extract_y


def balance_dataset(X, y):
    pos_idx, neg_idx = np.where(y == 1)[0], np.where(y == 0)[0]
    if len(pos_idx) > len(neg_idx):
        pos_idx = np.random.choice(pos_idx, size=len(neg_idx), replace=False)
    else:
        neg_idx = np.random.choice(neg_idx, size=len(pos_idx), replace=False)

    pos_X, pos_y = X[pos_idx], y[pos_idx]
    neg_X, neg_y = X[neg_idx], y[neg_idx]
    res_X = np.vstack([pos_X, neg_X])
    res_y = np.vstack([pos_y, neg_y])
    print(res_y.shape)
    logging.info(
        f"X: {res_X.shape} / y(pos/neg): {res_y.size}({sum(res_y==1)[0]}/{sum(res_y==0)[0]})"
    )
    return res_X, res_y

In [4]:
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s"
)
np.random.seed(42)

In [5]:
args = Arguments()
config = load_config(args.conf)
env = set_environment(args, config, model_name="inceptiontime")

In [6]:
pos_ds = np.vstack([load_npy(f) for f in tqdm(env["pos_path"][:2], ncols=75)])
print(pos_ds.shape)

100%|████████████████████████████████████████| 2/2 [02:36<00:00, 78.29s/it]


(23105, 3000, 3)


In [7]:
neg_ds = np.vstack([load_npy(f) for f in tqdm(env["neg_path"][:2], ncols=75)])
print(neg_ds.shape)

100%|███████████████████████████████████████| 2/2 [04:36<00:00, 138.15s/it]

(41099, 3000, 3)


In [8]:
pos_y, neg_y = np.ones((len(pos_ds), 1)), np.zeros((len(neg_ds), 1))

In [9]:
print(pos_ds.shape, pos_y.shape)
print(neg_ds.shape, neg_y.shape)

(23105, 3000, 3) (23105, 1)
(41099, 3000, 3) (41099, 1)


In [10]:
ds, y = np.vstack([pos_ds, neg_ds]), np.vstack([pos_y, neg_y])

In [11]:
cid, dt, X = ds.transpose(2, 0, 1)

In [12]:
X = scale_input(X)

In [13]:
train_cid, val_cid, test_cid = split_array(np.unique(cid), 0.2, config["seed"])

2026-03-19 23:00:11,383 [INFO] total(train/val/test): 600(384/96/120) cases


In [14]:
train_X, train_y = extract_dataset(X, y, np.isin(cid[:, 0], train_cid))
train_bal_X, train_bal_y = balance_dataset(train_X, train_y)
val_X, val_y = extract_dataset(X, y, np.isin(cid[:, 0], val_cid))
test_X, test_y = extract_dataset(X, y, np.isin(cid[:, 0], test_cid))

2026-03-19 23:00:23,038 [INFO] X: (40607, 3000, 1) / y(pos/neg): 40607(14988/25619)
2026-03-19 23:00:23,181 [INFO] X: (29976, 3000, 1) / y(pos/neg): 29976(14988/14988)


(40607, 1)
(29976, 1)


2026-03-19 23:00:23,541 [INFO] X: (10325, 3000, 1) / y(pos/neg): 10325(3649/6676)


(10325, 1)


2026-03-19 23:00:23,938 [INFO] X: (13272, 3000, 1) / y(pos/neg): 13272(4468/8804)


(13272, 1)
